In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
# Crear widgets
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_cr")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsfinalproject")

In [0]:
# Obtener valores de widgets
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

# Obtener ruta de archivo crimes.csv
ruta_crimes = f"abfss://{container}@{storageName}.dfs.core.windows.net/crimes.csv"

# Obtener ruta de archio months.csv
ruta_months = f"abfss://{container}@{storageName}.dfs.core.windows.net/months.csv"

In [0]:
# Crear DataFrame df_crimes infiriendo el schema.
df_crimes = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta_crimes)

In [0]:
# Crear DataFrame df_months infiriendo el schema.
df_months = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta_months)

In [0]:
# Cambiar nombres de columnas de df_crimes. Agregar columna ingestion_date.
df_crimes = df_crimes.select(
    col("ID").alias("crime_id"),
    col("NOMBRE_DELITO").alias("crime")
).withColumn("ingestion_date", current_timestamp())

In [0]:
# Cambiar nombres de columnas de df_months. Agregar columna ingestion_date.
df_months = df_months.select(
    col("ID").alias("month_id"),
    col("MES").alias("month_name")
).withColumn("ingestion_date", current_timestamp())

In [0]:
# Ingestar data en la tabla crimes.
df_crimes.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.crimes")

In [0]:
# Ingestar data en la tabla months.
df_months.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.months")